In [12]:
import sys
import plotly.graph_objects as go
from pathlib import Path
import pandas as pd
import numpy as np
import re
# --- Data Loading ---

files = [
    'prices_round_3_day_0.csv',
    'prices_round_3_day_1.csv',
    'prices_round_3_day_2.csv'
]

prices = []
for file in files:
    try:
        df = pd.read_csv(file, sep=';')
        day_match = re.search(r'day_(-?\d+)', file)
        if day_match:
            df['day'] = int(day_match.group(1))
        prices.append(df)
    except FileNotFoundError:
        print(f"Warning: {file} not found.")

df_total = pd.concat(prices, ignore_index=True)
df_total = df_total.sort_values(by=['day', 'timestamp']).reset_index(drop=True)

In [3]:
days = [0, 1, 2]
for day in days:
    # Filter for both products
    subset = df_total[(df_total['product'].isin(["HYDROGEL_PACK", "VELVETFRUIT_EXTRACT"])) & (df_total['day'] == day)].copy()
    
    if subset.empty:
        print(f"Skipping day {day}: No data found.")
        continue
    
    # Handle Mid Price
    subset['mid_price'] = subset['mid_price'].replace(0, np.nan)

    # Create the Interactive Plotly Figure
    fig = go.Figure()
    
    for product in ["HYDROGEL_PACK", "VELVETFRUIT_EXTRACT"]:
        prod_subset = subset[subset['product'] == product].copy()
        
        if prod_subset.empty:
            continue
        
        # Normalize price: divide by the first non-NaN mid_price
        first_price = prod_subset['mid_price'].dropna().iloc[0] if not prod_subset['mid_price'].dropna().empty else 1
        prod_subset['normalized_price'] = prod_subset['mid_price'] / first_price
        
        # Add trace for normalized price
        fig.add_trace(go.Scatter(x=prod_subset['timestamp'], y=prod_subset['normalized_price'], 
                                 name=f'{product} Normalized Price', mode='lines'))
    
    # Layout Customization
    fig.update_layout(
        title=f'Normalized Price Comparison: HYDROGEL_PACK vs VELVETFRUIT_EXTRACT - Day {day}',
        xaxis_title='Timestamp',
        yaxis_title='Normalized Price',
        legend_title='Products',
        template='plotly_white',
        hovermode='x unified'
    )
    
    fig.show()

In [22]:
# Normalized Price Spread Analysis
days = [0, 1, 2]

# Create the Interactive Plotly Figure outside the loop
fig = go.Figure()

for day in days:
    # Filter for both products
    subset = df_total[(df_total['product'].isin(["HYDROGEL_PACK", "VELVETFRUIT_EXTRACT"])) & (df_total['day'] == day)].copy()
    
    if subset.empty:
        print(f"Skipping day {day}: No data found.")
        continue
    
    # Handle Mid Price
    subset['mid_price'] = subset['mid_price'].replace(0, np.nan)

    # Compute normalized prices for each product
    hydro_df = None
    velvet_df = None
    
    for product in ["HYDROGEL_PACK", "VELVETFRUIT_EXTRACT"]:
        prod_subset = subset[subset['product'] == product].copy()
        
        if prod_subset.empty:
            continue
        
        # Normalize price: divide by the first non-NaN mid_price
        first_price = prod_subset['mid_price'].dropna().iloc[0] if not prod_subset['mid_price'].dropna().empty else 1
        prod_subset['normalized_price'] = prod_subset['mid_price'] / first_price
        
        if product == "HYDROGEL_PACK":
            hydro_df = prod_subset[['timestamp', 'normalized_price']].rename(columns={'normalized_price': 'hydro_norm'})
        else:
            velvet_df = prod_subset[['timestamp', 'normalized_price']].rename(columns={'normalized_price': 'velvet_norm'})
    
    # Merge the dataframes on timestamp
    if hydro_df is not None and velvet_df is not None:
        merged = pd.merge(hydro_df, velvet_df, on='timestamp', how='outer').sort_values('timestamp')
        merged['spread'] = merged['hydro_norm'] - merged['velvet_norm']
        
        # Add trace for spread
        fig.add_trace(go.Scatter(x=merged['timestamp'], y=merged['spread'], 
                                 name=f'Price Spread Day {day} (HYDROGEL_PACK - VELVETFRUIT_EXTRACT)', mode='lines'))
        
    else:
        print(f"Data missing for one or both products on day {day}.")

# Layout Customization
fig.update_layout(
    title='Normalized Price Spread: HYDROGEL_PACK - VELVETFRUIT_EXTRACT Over All Days',
    xaxis_title='Timestamp',
    yaxis_title='Spread',
    legend_title='Days',
    template='plotly_white',
    hovermode='x unified'
)

fig.show()

In [27]:
# Normalized Spread Changes Analysis (Across Days)
days = [0, 1, 2]

# First, compute global min and max of spreads across all days
all_spreads = []
for day in days:
    subset = df_total[(df_total['product'].isin(["HYDROGEL_PACK", "VELVETFRUIT_EXTRACT"])) & (df_total['day'] == day)].copy()
    if subset.empty:
        continue
    subset['mid_price'] = subset['mid_price'].replace(0, np.nan)
    hydro_df = None
    velvet_df = None
    for product in ["HYDROGEL_PACK", "VELVETFRUIT_EXTRACT"]:
        prod_subset = subset[subset['product'] == product].copy()
        if prod_subset.empty:
            continue
        if product == "HYDROGEL_PACK":
            hydro_df = prod_subset[['timestamp', 'mid_price']].rename(columns={'mid_price': 'hydro_price'})
        else:
            velvet_df = prod_subset[['timestamp', 'mid_price']].rename(columns={'mid_price': 'velvet_price'})
    if hydro_df is not None and velvet_df is not None:
        merged = pd.merge(hydro_df, velvet_df, on='timestamp', how='outer').sort_values('timestamp')
        merged['spread'] = merged['hydro_price'] - merged['velvet_price']
        all_spreads.extend(merged['spread'].dropna().tolist())

global_min = min(all_spreads) if all_spreads else 0
global_max = max(all_spreads) if all_spreads else 1

# Now, create the plot
fig = go.Figure()

for day in days:
    # Filter for both products
    subset = df_total[(df_total['product'].isin(["HYDROGEL_PACK", "VELVETFRUIT_EXTRACT"])) & (df_total['day'] == day)].copy()
    
    if subset.empty:
        print(f"Skipping day {day}: No data found.")
        continue
    
    # Handle Mid Price
    subset['mid_price'] = subset['mid_price'].replace(0, np.nan)

    # Compute raw prices for each product
    hydro_df = None
    velvet_df = None
    
    for product in ["HYDROGEL_PACK", "VELVETFRUIT_EXTRACT"]:
        prod_subset = subset[subset['product'] == product].copy()
        
        if prod_subset.empty:
            continue
        
        if product == "HYDROGEL_PACK":
            hydro_df = prod_subset[['timestamp', 'mid_price']].rename(columns={'mid_price': 'hydro_price'})
        else:
            velvet_df = prod_subset[['timestamp', 'mid_price']].rename(columns={'mid_price': 'velvet_price'})
    
    # Merge the dataframes on timestamp
    if hydro_df is not None and velvet_df is not None:
        merged = pd.merge(hydro_df, velvet_df, on='timestamp', how='outer').sort_values('timestamp')
        merged['spread'] = merged['hydro_price'] - merged['velvet_price']
        
        # Normalize the spread using global min-max
        if global_max != global_min:
            merged['normalized_spread'] = (merged['spread'] - global_min) / (global_max - global_min)
        else:
            merged['normalized_spread'] = 0
        
        # Add trace for normalized spread
        fig.add_trace(go.Scatter(x=merged['timestamp'], y=merged['normalized_spread'], 
                                 name=f'Normalized Spread Day {day} (HYDROGEL_PACK - VELVETFRUIT_EXTRACT)', mode='lines'))
        
    else:
        print(f"Data missing for one or both products on day {day}.")

# Layout Customization
fig.update_layout(
    title='Normalized Spread Changes Across Days: HYDROGEL_PACK - VELVETFRUIT_EXTRACT',
    xaxis_title='Timestamp',
    yaxis_title='Normalized Spread (0-1, Global Scale)',
    legend_title='Days',
    template='plotly_white',
    hovermode='x unified'
)

fig.show()

In [29]:
# Mean-Normalized Price Comparison
days = [0, 1, 2]
for day in days:
    # Filter for both products
    subset = df_total[(df_total['product'].isin(["HYDROGEL_PACK", "VELVETFRUIT_EXTRACT"])) & (df_total['day'] == day)].copy()
    
    if subset.empty:
        print(f"Skipping day {day}: No data found.")
        continue
    
    # Handle Mid Price
    subset['mid_price'] = subset['mid_price'].replace(0, np.nan)

    # Create the Interactive Plotly Figure
    fig = go.Figure()
    
    for product in ["HYDROGEL_PACK", "VELVETFRUIT_EXTRACT"]:
        prod_subset = subset[subset['product'] == product].copy()
        
        if prod_subset.empty:
            continue
        
        # Normalize price: divide by the mean mid_price
        mean_price = prod_subset['mid_price'].dropna().mean() if not prod_subset['mid_price'].dropna().empty else 1
        prod_subset['normalized_price'] = prod_subset['mid_price'] / mean_price
        
        # Add trace for normalized price
        fig.add_trace(go.Scatter(x=prod_subset['timestamp'], y=prod_subset['normalized_price'], 
                                 name=f'{product} Mean-Normalized Price', mode='lines'))
    
    # Layout Customization
    fig.update_layout(
        title=f'Mean-Normalized Price Comparison: HYDROGEL_PACK vs VELVETFRUIT_EXTRACT - Day {day}',
        xaxis_title='Timestamp',
        yaxis_title='Mean-Normalized Price',
        legend_title='Products',
        template='plotly_white',
        hovermode='x unified'
    )
    
    fig.show()